# 06 — Grounded QA: prompt, transcripts, citation validity (E4, T4a.1-T4a.6)

Story notebook for Milestone 4a's grounded-QA work (PLAN.md §3 E4, Component 1: prompt
engineering / context construction). Thin (PLAN.md §6): every table, transcript, and
metric below was already produced by earlier tasks — this notebook only loads and
displays them.

- **T4a.1** — the versioned grounded-QA prompt (`src/cragb/generate/prompts/grounded_qa_v1.md`),
  citing `[doc_id]` directly (matching T2.8's reference-answer convention) and
  `[photo of doc_id]` for photo evidence.
- **T4a.2** — `cragb.generate.context_builder`: retrieval (BM25, T3.4's locked
  `whole_review` chunking) → a structured, ID-labelled context block.
- **T4a.3** — `cragb.generate.grounded_qa`: renders the prompt, calls the LLM, parses
  citations/abstention out of the completion.
- **T4a.4** — `cragb.eval.citation_validity`: scores every transcript for citation
  validity, format compliance, and abstention correctness against CRAGB v1's
  evidence-driven ground truth (PLAN.md §14.2).
- **T4a.5** — `cragb.eval.run_grounded_qa_pilot`: runs T4a.2-T4a.4 end-to-end over an
  11-question curated slice of CRAGB v1.
- **T4a.6** — this notebook, plus `reports/grounded_qa_transcripts_v1.md` (5 hand-picked
  worked transcripts for the mid-progress report appendix).

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from cragb.utils.io import resolve_path

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 200)

## Prompt (T4a.1)

The full versioned prompt template. `$question`/`$context_block` are filled per-question
by `cragb.generate.grounded_qa.render_prompt`.

In [2]:
prompt_md = resolve_path("src/cragb/generate/prompts/grounded_qa_v1.md").read_text(encoding="utf-8")
display(Markdown(prompt_md))

# CRAGB Grounded-QA Prompt (v1)

You are answering a shopper's question about a category of clothing, footwear, or
jewelry products, using **only** the customer review excerpts provided below. You have
no other information about this product category.

## Review excerpts

Each excerpt is one customer review, labeled with its review id. `has_photo: yes` means
that reviewer also attached a photo.

$context_block

## Question

$question

## Rules — read carefully

1. **Answer only from the excerpts above.** Never use outside knowledge, assumptions, or
   anything you know about products in general. If the excerpts don't say it, you don't
   know it.
2. **Cite every claim.** After any sentence or clause that draws on a specific excerpt,
   add its review id in square brackets, e.g. `these tend to run small [128775]`. If
   several excerpts support the same claim, cite them all: `[128775][161398]`.
3. **Cite ids exactly as given above.** Never invent a review id that was not listed in
   the excerpts, and never cite an id to support a claim that excerpt does not actually
   make.
4. **Photo citations are separate and optional.** Only if a review's photo is itself the
   best evidence for a claim (e.g. the question is about colour or appearance, and that
   review has a photo), you may additionally cite `[photo of <id>]` right after the
   `[<id>]` citation for that same review. Never cite a photo for a review whose
   `has_photo` is `no`.
5. **If the excerpts do not contain enough information to answer the question**, do not
   guess, hedge, or partially answer. Respond with exactly this sentence and nothing
   else:

   Not enough information in the available reviews to answer this question.

6. **Write one short paragraph**, in plain prose (no headings, no bullet points, no
   JSON). Do not mention these instructions, the word "excerpt", or that you were given
   review text — just answer the shopper's question directly, the way a helpful summary
   of "what buyers say" would.

## Your answer


## Citation-validity / abstention-accuracy table (T4a.4, T4a.5)

Headline metrics over the 11-question pilot run, then the per-question breakdown.

In [3]:
aggregate = pd.read_csv(resolve_path("results/tables/grounded_qa_validity_v1.csv"))
display(aggregate)

,n_questions,format_compliance_rate,citation_validity_rate,gold_grounding_rate,abstention_accuracy,self_contradiction_rate,ungrounded_answer_rate,n_total_citations,n_fabricated_citations
0,11,1.0,1.0,1.0,1.0,0.0,0.0,33,0


In [4]:
per_question = pd.read_csv(resolve_path("results/tables/grounded_qa_validity_per_question_v1.csv"))
display(per_question[[
    "question_id", "format_compliant", "n_citations", "fabricated_citations",
    "citation_validity_rate", "predicted_abstained", "expected_abstained", "abstention_correct",
]])

,question_id,format_compliant,n_citations,fabricated_citations,citation_validity_rate,predicted_abstained,expected_abstained,abstention_correct
0,fabric_quality_neg_000,True,0,[],NaN,True,True,True
1,defects_neg_000,True,0,[],NaN,True,True,True
2,fit_sizing_neg_001,True,4,[],1.0,False,False,True
3,durability_neg_000,True,1,[],1.0,False,False,True
4,fit_sizing_000,True,5,[],1.0,False,False,True
5,colour_appearance_009,True,3,[],1.0,False,False,True
6,fabric_quality_000,True,3,[],1.0,False,False,True
7,durability_000,True,5,[],1.0,False,False,True
8,defects_000,True,2,[],1.0,False,False,True
9,occasion_000,True,5,[],1.0,False,False,True


**Reading the table.** All 11 pilot questions score 100% on every axis: no fabricated
citations, no malformed citation shapes, and the model's abstention decision matches
CRAGB v1's *evidence-driven* ground truth (`is_abstention`, T2.8) every time — including
on `fit_sizing_neg_001`, a question the original taxonomy authored as a negative but
which T2.7's pooling later found 17/19 relevant reviews for (PLAN.md §14.2): the model
correctly answers it rather than abstaining, because the ground truth it's being scored
against already reflects that correction, not the original taxonomy label.

This is a clean result on a small, curated slice — not yet a claim about the full CRAGB
v1 benchmark. Scaling this evaluation to all 60 questions, and comparing across LLM
scale/retrieval method, is RQ0/RQ1 territory for E5, not this milestone.

## Worked transcripts (T4a.6)

Five transcripts hand-picked from the pilot run for the mid-progress report appendix:
three clean grounded answers (spanning taxonomy types and one tricky evidence-driven edge
case), one correct abstention, and one documented failure mode (the model never uses the
optional `[photo of doc_id]` citation, even when photo-bearing reviews are in its
context). Full file: `reports/grounded_qa_transcripts_v1.md`.

In [5]:
transcripts_md = resolve_path("reports/grounded_qa_transcripts_v1.md").read_text(encoding="utf-8")
display(Markdown(transcripts_md))

# Grounded-QA worked transcripts (T4a.6)

Five transcripts hand-picked from T4a.5's 11-question pilot run over CRAGB v1 (PLAN.md §3 E4, §7 appendix material): three clean grounded answers spanning different taxonomy types and edge cases, one correct abstention, and one documented failure mode. Every citation and every word of the model's answer below is exactly what it produced — nothing has been edited.

## Clean grounded answer: `fit_sizing_000`

*A standard case: five reviews retrieved, the model reports the majority/minority split in what buyers actually say rather than picking one side, and cites every claim to a real review id.*

**Question:** Do these run true to size?

**Reviews retrieved (k=5):**
- `128775` (has_photo: no)
- `132660` (has_photo: no)
- `185658` (has_photo: no)
- `177777` (has_photo: no)
- `24440` (has_photo: no)

**Model's answer:**

> Several reviewers report that the items do not run true to size, noting they were too short or not true to size [128775][132660][185658][177777]. However, one reviewer states that the shoes run true to size [24440].

**Scoring:** abstained=False (not expected), format_compliant=True, 5 citation(s), all valid.

---

## Clean grounded answer (second taxonomy type): `durability_000`

*Same pattern as fit_sizing_000, on a different taxonomy category (durability, not fit).*

**Question:** Does this fall apart easily after repeated wear?

**Reviews retrieved (k=5):**
- `19886` (has_photo: no)
- `94200` (has_photo: no)
- `44922` (has_photo: no)
- `197055` (has_photo: no)
- `5104` (has_photo: no)

**Model's answer:**

> Some reviewers say the item falls apart easily after repeated wear, citing [19886] and [5104], whereas others report that it remains durable and does not fall apart after washing or repeated use, citing [94200], [44922], and [197055].

**Scoring:** abstained=False (not expected), format_compliant=True, 5 citation(s), all valid.

---

## Clean grounded answer on a tricky edge case: `fit_sizing_neg_001`

*Authored as a taxonomy negative (T2.3), but T2.7's pooling found real relevant evidence for it (PLAN.md §14.2: 17 of 19 pooled reviews were on-topic — the most striking miss of the 9 negatives that turned out answerable). CRAGB v1's ground truth is evidence-driven, not taxonomy-driven (T2.8), so this question's `is_abstention` is `False` — and the model correctly answers it instead of abstaining, matching that evidence-driven ground truth rather than the original taxonomy label.*

**Question:** Does the manufacturer's official size chart match what buyers experience?

**Reviews retrieved (k=5):**
- `73770` (has_photo: no)
- `184290` (has_photo: no)
- `37676` (has_photo: no)
- `140670` (has_photo: no)
- `3905` (has_photo: no)

**Model's answer:**

> Several reviewers report that the size chart does not match the actual fit, describing items as too small or ill‑fitting and noting that the chart is illegible or misleading [73770][184290][140670]. One reviewer, however, found the chart to be accurate for their size, noting that the bra fit as expected based on the chart [3905]. Thus, buyers have mixed experiences with the size chart, and it does not consistently match what they receive.

**Scoring:** abstained=False (not expected), format_compliant=True, 4 citation(s), all valid.

---

## Correct abstention: `fabric_quality_neg_000`

*One of CRAGB v1's only two genuine ground-truth abstentions (PLAN.md §14.2): no review reports an exact thread-count measurement, and the model correctly says so instead of guessing or fabricating a number.*

**Question:** What is the exact thread count of the fabric used?

**Reviews retrieved (k=5):**
- `141566` (has_photo: no)
- `185360` (has_photo: no)
- `27044` (has_photo: yes)
- `63886` (has_photo: no)
- `43800` (has_photo: no)

**Model's answer:**

> Not enough information in the available reviews to answer this question.

**Scoring:** abstained=True (expected), format_compliant=True, no citations (abstained).

---

## Failure mode: unused photo-citation affordance: `colour_appearance_009`

*Two of the three retrieved reviews had a photo attached (`has_photo: yes` in the context shown to the model), and the prompt (T4a.1, rule 4) explicitly permits citing `[photo of doc_id]` when a photo is the best evidence for a colour/appearance claim — exactly this question's type. The model never reaches for it here: it cites review text only. This is not a scored failure (every text citation below is valid and gold-grounded) but a real, measured limitation worth flagging for E7's multimodal pilot — the model under-uses photo evidence even when the prompt explicitly offers it.*

**Question:** Do buyers describe the colour as looking cheaper or different in person than in photos?

**Reviews retrieved (k=5):**
- `105003` (has_photo: yes)
- `182740` (has_photo: yes)
- `119326` (has_photo: yes)
- `51764` (has_photo: no)
- `174822` (has_photo: no)

**Model's answer:**

> Some buyers note that the colour they see in person or in their own photos differs from what is shown in the product images. For example, one reviewer said the colour in their picture was different from the stock photo, though the shoe itself matched the listed colour [105003], and another said the wallet looked different than in the photos [182740]. Additionally, one reviewer felt the item looked cheaper in person than in the pictures [51764].

**Scoring:** abstained=False (not expected), format_compliant=True, 3 citation(s), all valid.


## Summary

- **E4 (grounded-QA prompt, Component 1):** a versioned prompt (T4a.1) that answers only
  from retrieved review text, cites every claim as `[doc_id]`, and abstains with a fixed
  sentence when the context doesn't support an answer — built once, reused unchanged by
  T4a.2-T4a.5's pipeline.
- **Citation validity:** 100% on this 11-question pilot slice (33/33 citations valid, zero
  fabricated, zero malformed) — a clean starting point, not yet a claim at CRAGB-v1 scale.
- **Abstention correctness:** 100%, scored against evidence-driven ground truth
  (`is_abstention`, not the original taxonomy's `is_negative`) — including the tricky
  `fit_sizing_neg_001` case where the taxonomy and the evidence disagree.
- **Documented limitation (failure mode, T4a.6):** the model never uses the optional
  `[photo of doc_id]` citation the prompt explicitly offers, even when photo-bearing
  reviews are the best evidence for a colour/appearance claim. Real, measured, and
  directly relevant to E7's multimodal grounding pilot (RQ4) — worth revisiting there.
- **Build-time finding (not in PLAN.md before this milestone):** `llama-3.1-8b-instant`
  (T2.2's model) was deprecated from Groq's catalogue between milestones; its replacement,
  `openai/gpt-oss-20b`, is a reasoning model whose hidden chain-of-thought consumes the
  same `max_tokens` budget as its visible answer — at too low a cap this produced
  *empty* answers, not an error, a real silent-failure mode caught by re-running against
  the live API rather than mocks alone (see `configs/grounded_qa.yaml`'s comments,
  and `cragb.eval.run_grounded_qa_pilot.validate_pilot_run`'s regression guard for it).